# Offline localization test: do the code graphs and embeddings find the code that the fix changes?

**Question.** Start from the names an issue mentions. How often does each tool return a function that the correct fix changes
("gold" function), and how much does the agent have to read to get there? Compared with plain `grep` and with chance.
No model is involved: everything runs on the competition's graph and embedding files plus the original source at each task's commit.

**What each method copies (checked in `swegemma` 0.2.11 source):**
- `get_code_neighbors(node)`: every node with an edge to or from `node`, **sorted alphabetically**, cut to 50
  (`swegemma/graph/graph_utils.py`, `get_neighbor`). Not ranked, so we measure "is a gold function in the set" and the set size.
- `search_similar_code(query, k=10)`: if `query` names a node, raw cosine similarity between the stored 256-number vectors, top-k
  without the node itself (`graph/retrieval_utils.py`, `get_similar_nodes`). No model: `embed()` only looks up stored vectors.
  If `query` is not a node (free text), it falls back to keyword scoring: +3 per word in the node name, +1 per word in its code,
  +0.5 if not a test.
- Names resolve like the harness: exact id, then suffix on a `.` boundary (shortest wins), then case-insensitive (`resolve_node_name`).
- `grep`: functions in the package source at that commit whose code contains the name as a whole word: roughly what
  `grep -rn name` would show the agent, counted per function.

**Gold functions.** For each changed non-test `.py` file in the task's `patch`, read the file at `base_commit` (partial clones in
`reference/repos/`), list every `def` / `class` with Python's `ast` (functions nested inside functions count as their enclosing
function), and map each removed line to the innermost one containing it. For added-only lines, use the last code line before the
insertion and the indentation of the new code (a new method → its class; new lines inside a function → that function; a new
top-level function → no gold function, file level only). Gold is independent of the graph.

**Found while building this:** the graphs contain **no `async def` functions** (0 of 313 in the changed files), and no functions
nested in functions. 99.6% of normal functions and methods are there. So the graph tools cannot return an async gold function.

In [1]:
import json, re, ast, subprocess, pickle
from collections import Counter
from math import comb
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('../..')
DATA = ROOT / 'competition_data'
REPOS = ROOT / 'reference' / 'repos'
PACKAGE_DIR = {'fastapi': 'fastapi', 'rich': 'rich', 'requests': 'src/requests', 'httpx': 'httpx'}

tasks = [json.loads(line) for line in open(DATA / 'tasks.jsonl', encoding='utf-8')]
by_id = {t['instance_id']: t for t in tasks}

def repo_short(t):
    return t['repo'].split('/')[-1]

def snapshot(t):
    # graph / embedding files are named <repo short name>_<base_commit>
    return f"{repo_short(t)}_{t['base_commit']}"

def is_test_node(name):
    # the harness's own test check (retrieval_utils.py keyword fallback)
    return name.startswith(('test.', 'tests.')) or '.test_' in name

def is_test_path(path):
    return path.startswith(('tests/', 'test/')) or '/tests/' in path or Path(path).name.startswith('test_')

print(len(tasks), 'tasks,', len({snapshot(t) for t in tasks}), 'snapshots')

129 tasks, 127 snapshots


In [2]:
# Python source helpers: module name of a file, and every def / class in it with its line range
def module_of(path):
    parts = list(Path(path).with_suffix('').parts)
    if parts[0] == 'src':
        parts = parts[1:]
    if parts[-1] == '__init__':
        parts = parts[:-1]
    return '.'.join(parts)

def def_ranges(src):
    # (qualified name, first line, last line, column); defs nested inside a function are skipped,
    # so a line inside them belongs to the enclosing function
    out = []
    def walk(node, prefix, in_function):
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                q = f'{prefix}{child.name}'
                if not in_function:
                    start = min([child.lineno] + [d.lineno for d in child.decorator_list])
                    out.append((q, start, child.end_lineno, child.col_offset))
                walk(child, q + '.', in_function or not isinstance(child, ast.ClassDef))
            else:
                walk(child, prefix, in_function)
    walk(ast.parse(src), '', False)
    return out

def git(t, *args):
    return subprocess.run(['git', f'--git-dir={REPOS / (repo_short(t) + ".git")}', *args],
                          capture_output=True, text=True, encoding='utf-8')

def source_at(t, path):
    out = git(t, 'show', f"{t['base_commit']}:{path}")
    return out.stdout if out.returncode == 0 else None   # None: the file is new in the fix

In [3]:
# Source index per snapshot: every def / class in the package folder at base_commit, with its code (for grep).
# Uses one sparse worktree per repo (package folder only) and caches the result; the first run takes ~5 min.
INDEX_DIR = REPOS / 'index'
INDEX_DIR.mkdir(exist_ok=True)

def source_index(t):
    cache = INDEX_DIR / f'{snapshot(t)}.pkl'
    if cache.exists():
        return pickle.load(open(cache, 'rb'))
    repo, pkg = repo_short(t), PACKAGE_DIR[repo_short(t)]
    wt = REPOS / f'{repo}-src'
    if not wt.exists():
        git(t, 'worktree', 'add', '-q', '--no-checkout', str(wt))
        subprocess.run(['git', '-C', str(wt), 'sparse-checkout', 'set', '--cone', pkg], check=True)
    subprocess.run(['git', '-C', str(wt), 'checkout', '-q', '--detach', t['base_commit']], check=True)
    index = {}
    for f in sorted((wt / pkg).rglob('*.py')):
        rel = f.relative_to(wt).as_posix()
        src = f.read_text(encoding='utf-8')
        lines = src.splitlines()
        for q, start, end, _ in def_ranges(src):
            index[f'{module_of(rel)}.{q}'] = '\n'.join(lines[start - 1:end])
    pickle.dump(index, open(cache, 'wb'))
    return index

src_index = {}
for t in tasks:
    if snapshot(t) not in src_index:
        src_index[snapshot(t)] = source_index(t)
print(len(src_index), 'snapshots indexed; median defs per snapshot:', int(np.median([len(v) for v in src_index.values()])))

127 snapshots indexed; median defs per snapshot: 426


In [4]:
# Graph and embeddings for one snapshot (cached: two pairs of tasks share a snapshot)
_cache = {}

def load_snapshot(name):
    if name not in _cache:
        g = json.load(open(DATA / 'graphs' / f'{name}.json', encoding='utf-8'))
        text = {n['id']: n.get('text') or '' for n in g['nodes']}
        nbrs = {n: set() for n in text}
        for e in g['edges']:                       # both directions, like get_neighbor (out + in edges)
            nbrs.setdefault(e['source'], set()).add(e['target'])
            nbrs.setdefault(e['target'], set()).add(e['source'])
        npz = np.load(DATA / 'embeddings' / f'{name}.npz')
        names = list(npz.files)
        vecs = np.stack([npz[n] for n in names]).astype(np.float32)
        unit = lambda v: v / np.maximum(np.linalg.norm(v, axis=1, keepdims=True), 1e-12)
        _cache[name] = {'text': text, 'nbrs': nbrs, 'emb_names': names, 'emb_index': {n: i for i, n in enumerate(names)},
                        'unit': unit(vecs),                          # what the harness uses
                        'unit_centered': unit(vecs - vecs.mean(0))}  # not used by the harness: mean vector removed first
    return _cache[name]

s = load_snapshot(snapshot(tasks[0]))
print(len(s['text']), 'nodes,', sum(map(len, s['nbrs'].values())) // 2, 'undirected links,', len(s['emb_names']), 'vectors')

4263 nodes, 1208 undirected links, 4263 vectors


In [5]:
# What the graph leaves out: for the package source at each commit, which kinds of def are graph nodes?
kinds = Counter()
for t in {snapshot(t): t for t in tasks}.values():
    graph_ids = load_snapshot(snapshot(t))['text']
    for qid, code in src_index[snapshot(t)].items():
        first = code.lstrip().split('\n', 1)[0]
        kind = ('async def' if re.match(r'(@.*\n\s*)*async\s+def', code.lstrip()) or 'async def' in first
                else 'class' if re.match(r'(@[^\n]*\n\s*)*class\b', code.lstrip()) else 'def')
        kinds[(kind, qid in graph_ids)] += 1
pd.DataFrame([{'kind': k, 'in graph': v_in, 'count': n} for (k, v_in), n in kinds.items()]) \
  .pivot_table(index='kind', columns='in graph', values='count', fill_value=0)

in graph,False,True
kind,,
async def,1918.0,0.0
class,76.0,16365.0
def,0.0,58351.0


In [6]:
# Gold functions: which defs does each correct fix change? (independent of the graph)
def parse_patch(patch):
    # list of (old path, hunks); a hunk is a list of (kind, old_line_no, text) with kind in ' ', '-', '+'
    files, path, hunk, old_no = [], None, None, 0
    for line in patch.splitlines():
        if line.startswith('--- '):
            path = line[6:] if line.startswith('--- a/') else None
        elif line.startswith('+++ '):
            files.append((path or line[6:], []))
        elif line.startswith('@@'):
            old_no = int(re.match(r'@@ -(\d+)', line).group(1))
            hunk = []
            files[-1][1].append(hunk)
        elif hunk is not None and line[:1] in (' ', '-', '+'):
            hunk.append((line[0], old_no, line[1:]))
            if line[0] != '+':
                old_no += 1
    return files

def gold_for_task(t):
    gold, gold_files, unmapped = set(), set(), 0
    for path, hunks in parse_patch(t['patch']):
        if not path.endswith('.py') or is_test_path(path):
            continue
        gold_files.add(module_of(path))
        src = source_at(t, path)
        if src is None:                            # new file: nothing old to find
            continue
        old_lines, ranges, mod = src.splitlines(), def_ranges(src), module_of(path)

        def innermost(line_no, max_col=None):
            hits = [r for r in ranges if r[1] <= line_no <= r[2] and (max_col is None or r[3] < max_col)]
            return f'{mod}.{min(hits, key=lambda r: r[2] - r[1])[0]}' if hits else None

        for hunk in hunks:
            found = [innermost(no) for kind, no, _ in hunk if kind == '-']
            i = 0
            while i < len(hunk):                   # each run of added lines
                if hunk[i][0] != '+':
                    i += 1
                    continue
                j = i
                while j < len(hunk) and hunk[j][0] == '+':
                    j += 1
                added = [txt for _, _, txt in hunk[i:j] if txt.strip()]
                if added:
                    indent = len(added[0]) - len(added[0].lstrip())
                    anchor = hunk[i][1] - 1        # old line just before the insertion
                    while anchor > 0 and not old_lines[anchor - 1].strip():
                        anchor -= 1
                    found.append(innermost(anchor, max_col=indent) if anchor > 0 else None)
                i = j
            gold |= {f for f in found if f}
            unmapped += sum(f is None for f in found)
    return gold, gold_files, unmapped

for tid in ['requests_6629', 'requests_6589', 'fastapi_13537']:
    print(tid, gold_for_task(by_id[tid]))

requests_6629 ({'requests.exceptions.JSONDecodeError'}, {'requests.exceptions'}, 0)
requests_6589 ({'requests.utils.super_len'}, {'requests.utils'}, 0)
fastapi_13537 ({'fastapi.dependencies.utils._extract_form_body'}, {'fastapi.dependencies.utils'}, 0)


In [7]:
gold_rows = []
for t in tasks:
    gold, gold_files, unmapped = gold_for_task(t)
    graph_ids = load_snapshot(snapshot(t))['text']
    gold_rows.append({'task': t['instance_id'], 'repo': repo_short(t), 'gold': gold, 'gold_files': gold_files,
                      'n_gold': len(gold), 'n_gold_in_graph': sum(g in graph_ids for g in gold), 'unmapped_edits': unmapped})
df_gold = pd.DataFrame(gold_rows)

has_gold = df_gold['n_gold'] > 0
print('tasks with >= 1 gold function:', has_gold.sum(), 'of', len(df_gold))
print('... all gold functions in the graph:', (has_gold & (df_gold['n_gold_in_graph'] == df_gold['n_gold'])).sum(),
      '| some missing:', (has_gold & (df_gold['n_gold_in_graph'].between(1, df_gold['n_gold'] - 1))).sum(),
      '| none in the graph:', (has_gold & (df_gold['n_gold_in_graph'] == 0)).sum())
df_gold.groupby('repo')[['n_gold', 'n_gold_in_graph']].sum()

tasks with >= 1 gold function: 122 of 129
... all gold functions in the graph: 106 | some missing: 11 | none in the graph: 5


,n_gold,n_gold_in_graph
repo,,
fastapi,373,327
httpx,21,16
requests,14,14
rich,116,116


In [8]:
# Names in the issue text, resolved the harness's way: exact, then suffix on a '.' boundary, then case-insensitive
def shortest(cands):
    return min(cands, key=lambda x: (len(x.split('.')), len(x), x)) if cands else None

def resolve(name, ids, ids_lower):
    # ids: list of ids; ids_lower: lower-cased id -> id
    for q in dict.fromkeys([name, name.rsplit('.', 1)[-1]]):     # 'router.include_router' also tries 'include_router'
        if q in ids_lower.values():
            return q
        hit = shortest([n for n in ids if n.endswith('.' + q)])
        if hit:
            return hit
        low = q.lower()
        if low in ids_lower:
            return ids_lower[low]
        hit = shortest([ids_lower[n] for n in ids_lower if n.endswith('.' + low)])
        if hit:
            return hit
    return None

STOP = {'self', 'cls', 'None', 'True', 'False', 'str', 'int', 'dict', 'list', 'type', 'object', 'value', 'name', 'data',
        'and', 'the', 'for', 'not', 'with', 'from', 'import', 'return', 'def', 'class', 'print', 'http', 'https', 'com'}

def issue_symbols(text):
    # code-looking words (snake_case, CamelCase, dotted) anywhere, plus any word inside `backticks`
    span_tokens = [tok for s in re.findall(r'`([^`\n]+)`', text) for tok in re.findall(r'[A-Za-z_][\w.]*', s)]
    in_span = set(span_tokens)
    cands = []
    for w in span_tokens + re.findall(r'\b[A-Za-z_][\w.]*\b', text):
        w = w.strip('.')
        looks_like_code = ('_' in w or '.' in w or re.search(r'[a-z][A-Z]', w) or re.match(r'[A-Z][a-z]+[A-Z]', w))
        if len(w) >= 3 and w not in STOP and (looks_like_code or w in in_span):
            cands.append(w)
    return list(dict.fromkeys(cands))          # keep order, drop repeats

def resolve_all(names, ids):
    ids = list(ids)
    lower = {n.lower(): n for n in ids}
    return list(dict.fromkeys(r for r in (resolve(n, ids, lower) for n in names) if r))

query_rows = []
for t in tasks:
    names = issue_symbols(t['problem_statement'])
    query_rows.append({'task': t['instance_id'], 'names': names,
                       'graph_queries': resolve_all(names, load_snapshot(snapshot(t))['text']),   # what the graph tools get
                       'src_queries': resolve_all(names, src_index[snapshot(t)])})                 # what grep / reading gets
df_q = pd.DataFrame(query_rows)
df_q['n_graph_q'] = df_q['graph_queries'].str.len()
df_q['n_src_q'] = df_q['src_queries'].str.len()
print('tasks where >= 1 issue name is a graph node:', (df_q['n_graph_q'] > 0).sum(),
      '| a def in the package source:', (df_q['n_src_q'] > 0).sum(), '| either:', ((df_q['n_graph_q'] + df_q['n_src_q']) > 0).sum())

tasks where >= 1 issue name is a graph node: 64 | a def in the package source: 62 | either: 64


In [9]:
# The methods: each returns the nodes the agent would see for this task's names
def embed_topk(s, q, k, key='unit'):
    i = s['emb_index'].get(q)
    if i is None:
        return []
    sims = s[key] @ s[key][i]
    sims[i] = -np.inf                              # the harness leaves the query node out
    return [s['emb_names'][j] for j in np.argsort(-sims)[:k]]

def neighbours(s, q):
    return sorted(s['nbrs'].get(q, ()))[:50]       # alphabetical, cut to 50, like get_neighbor

def grep_hits(index, q):
    word = re.compile(r'\b' + re.escape(q.rsplit('.', 1)[-1]) + r'\b')
    return [n for n, code in index.items() if n != q and word.search(code)]

def keyword_fallback(s, text, k):
    tokens = [tok for tok in re.findall(r'[a-z0-9_]+', text.lower()) if len(tok) >= 2]
    scores = []
    for n, txt in s['text'].items():
        nl, tl = n.lower(), txt.lower()
        sc = sum(3.0 * (tok in nl) + 1.0 * (tok in tl) for tok in tokens)
        if sc > 0:
            scores.append((-(sc + 0.5 * (not is_test_node(nl))), n))
    return [n for _, n in sorted(scores)[:k]]

def module_hit(nodes, gold_files):
    return any(any(n == m or n.startswith(m + '.') for m in gold_files) for n in nodes)

In [10]:
rows = []
for t, g, q in zip(tasks, gold_rows, query_rows):
    s, index = load_snapshot(snapshot(t)), src_index[snapshot(t)]
    gold, files, gq, sq = g['gold'], g['gold_files'], q['graph_queries'], q['src_queries']
    title = t['problem_statement'].strip().splitlines()[0][:200]

    emb5 = {n for x in gq for n in embed_topk(s, x, 5)}
    emb10 = {n for x in gq for n in embed_topk(s, x, 10)}
    emb10c = {n for x in gq for n in embed_topk(s, x, 10, 'unit_centered')}
    nbr = {n for x in gq for n in neighbours(s, x)}
    grp = {n for x in sq for n in grep_hits(index, x)}
    kw10 = set(keyword_fallback(s, title, 10))
    named = set(gq) | set(sq)
    n_pool = len(index)
    n_gold_pool = len(gold & set(index))

    rows.append({
        'task': t['instance_id'], 'repo': g['repo'], 'n_gold': len(gold), 'n_gold_in_graph': g['n_gold_in_graph'],
        'n_graph_q': len(gq), 'n_src_q': len(sq),
        'named_in_issue': bool(named & gold),
        'emb_top5': bool(emb5 & gold), 'emb_top10': bool(emb10 & gold), 'n_emb5': len(emb5), 'n_emb10': len(emb10),
        'neighbours': bool(nbr & gold), 'n_neighbours': len(nbr),
        'grep': bool(grp & gold), 'n_grep': len(grp),
        'keyword_top10': bool(kw10 & gold),
        'named_or_emb10': bool((named | emb10) & gold), 'named_or_nbr': bool((named | nbr) & gold),
        'named_or_grep': bool((named | grp) & gold),
        'random10': 1 - comb(n_pool - n_gold_pool, 10) / comb(n_pool, 10) if n_gold_pool else 0.0,
        'n_pool': n_pool, 'n_gold_pool': n_gold_pool,
        'raw_vs_centred_overlap': (np.mean([len(set(embed_topk(s, x, 10)) & set(embed_topk(s, x, 10, 'unit_centered'))) / 10
                                            for x in gq]) if gq else np.nan),
        # file level: does any returned node sit in a file the fix changes?
        'file_named': module_hit(named, files), 'file_emb10': module_hit(emb10, files),
        'file_nbr': module_hit(nbr, files), 'file_grep': module_hit(grp, files), 'file_kw10': module_hit(kw10, files),
        # not what the tool does: the same vectors with the mean removed first
        'emb_top10_centered': bool(emb10c & gold),
        # how much of each embedding top-10 is just the query's graph neighbourhood (raw, and centred)
        'emb10_share_nbrs': np.mean([len(set(embed_topk(s, x, 10)) & s['nbrs'].get(x, set())) / 10 for x in gq]) if gq else np.nan,
        'emb10c_share_nbrs': (np.mean([len(set(embed_topk(s, x, 10, 'unit_centered')) & s['nbrs'].get(x, set())) / 10 for x in gq])
                              if gq else np.nan),
    })
df_loc = pd.DataFrame(rows)
print(df_loc.shape)

(129, 31)


In [11]:
# Main table: tasks with >= 1 gold function and >= 1 issue name that resolves (in the graph or the source)
ev = df_loc[(df_loc['n_gold'] > 0) & ((df_loc['n_graph_q'] + df_loc['n_src_q']) > 0)]
print(f'{len(ev)} of {len(df_loc)} tasks evaluated; graph tools got >= 1 name in {(ev["n_graph_q"] > 0).sum()} of them')

methods = {'named in issue (no tool)': ('named_in_issue', ev['n_graph_q'] + ev['n_src_q']),
           'embeddings top-5': ('emb_top5', ev['n_emb5']),
           'embeddings top-10': ('emb_top10', ev['n_emb10']),
           'graph neighbours': ('neighbours', ev['n_neighbours']),
           'grep': ('grep', ev['n_grep']),
           'keyword fallback, issue title': ('keyword_top10', pd.Series(10, index=ev.index)),
           'named + embeddings top-10': ('named_or_emb10', ev['n_graph_q'] + ev['n_src_q'] + ev['n_emb10']),
           'named + neighbours': ('named_or_nbr', ev['n_graph_q'] + ev['n_src_q'] + ev['n_neighbours']),
           'named + grep': ('named_or_grep', ev['n_graph_q'] + ev['n_src_q'] + ev['n_grep']),
           'random 10 functions': ('random10', pd.Series(10, index=ev.index))}
def chance(n_pool, n_gold, k):
    # probability that k functions picked at random from the package include >= 1 gold function
    k = int(min(k, n_pool))
    return 1 - comb(n_pool - n_gold, k) / comb(n_pool, k) if n_gold and k else 0.0

summary = pd.DataFrame({'finds a gold function (share of tasks)': [ev[c].mean() for c, _ in methods.values()],
                        'functions returned (median)': [n.median() for _, n in methods.values()],
                        # chance level for the same number of functions, task by task
                        'random pick of the same size': [np.mean([chance(p, g, k) for p, g, k in
                                                                  zip(ev['n_pool'], ev['n_gold_pool'], n)])
                                                         for _, n in methods.values()]},
                       index=list(methods)).round(2)
summary

63 of 129 tasks evaluated; graph tools got >= 1 name in 63 of them


,finds a gold function (share of tasks),functions returned (median),random pick of the same size
named in issue (no tool),0.46,4.0,0.06
embeddings top-5,0.17,10.0,0.11
embeddings top-10,0.22,20.0,0.16
graph neighbours,0.25,12.0,0.15
grep,0.51,73.0,0.28
"keyword fallback, issue title",0.40,10.0,0.07
named + embeddings top-10,0.56,24.0,0.18
named + neighbours,0.56,18.0,0.18
named + grep,0.71,75.0,0.30
random 10 functions,0.07,10.0,0.07


In [12]:
# File level (does the method point at a file the fix changes?), and the function-level results by repo
print(ev[['file_named', 'file_emb10', 'file_nbr', 'file_grep', 'file_kw10']].mean().round(2), '\n')
ev.groupby('repo')[['named_in_issue', 'emb_top10', 'neighbours', 'grep', 'random10']].mean().round(2) \
  .assign(tasks=ev.groupby('repo').size())

file_named    0.62
file_emb10    0.68
file_nbr      0.67
file_grep     0.83
file_kw10     0.57
dtype: float64 



,named_in_issue,emb_top10,neighbours,grep,random10,tasks
repo,,,,,,
fastapi,0.37,0.32,0.26,0.61,0.10,38
httpx,1.00,0.00,1.00,0.00,0.00,1
requests,0.67,0.00,0.00,0.33,0.04,9
rich,0.53,0.13,0.33,0.40,0.03,15


In [13]:
# Only tasks whose gold functions are all in the graph (so the async gap can't explain a miss)
ev_in = ev[ev['n_gold_in_graph'] == ev['n_gold']]
print(len(ev_in), 'tasks')
ev_in[['named_in_issue', 'emb_top10', 'neighbours', 'grep', 'keyword_top10', 'random10']].mean().round(2)

52 tasks


named_in_issue    0.44
emb_top10         0.17
neighbours        0.21
grep              0.46
keyword_top10     0.38
random10          0.04
dtype: float64

In [14]:
# Is embedding search just the graph neighbourhood in disguise? And do centred vectors do better?
print('share of an embedding top-10 that are direct graph neighbours of the query: raw',
      round(ev['emb10_share_nbrs'].mean(), 2), '| centred', round(ev['emb10c_share_nbrs'].mean(), 2))
print('gold function in top-10: raw (the tool)', round(ev['emb_top10'].mean(), 2),
      '| centred vectors (not the tool)', round(ev['emb_top10_centered'].mean(), 2),
      '| raw and centred top-10 share', round(ev['raw_vs_centred_overlap'].mean(), 2), 'of their results')
only_emb = ev[ev['emb_top10'] & ~ev['named_in_issue'] & ~ev['neighbours'] & ~ev['grep']]
print('tasks where only the embeddings find a gold function:', len(only_emb), list(only_emb['task']))
only_graph = ev[(ev['emb_top10'] | ev['neighbours']) & ~ev['named_in_issue'] & ~ev['grep']]
print('tasks where a graph tool finds one that neither the issue nor grep does:', len(only_graph), list(only_graph['task']))

share of an embedding top-10 that are direct graph neighbours of the query: raw 0.07 | centred 0.07
gold function in top-10: raw (the tool) 0.22 | centred vectors (not the tool) 0.22 | raw and centred top-10 share 0.87 of their results
tasks where only the embeddings find a gold function: 2 ['fastapi_15030', 'fastapi_9555']
tasks where a graph tool finds one that neither the issue nor grep does: 3 ['fastapi_15030', 'fastapi_9555', 'rich_3718']


In [15]:
# Keyword fallback works without any name in the issue, so it can be scored on every task with a gold function
all_gold = df_loc[df_loc['n_gold'] > 0]
print(len(all_gold), 'tasks: keyword fallback (issue title) finds a gold function in',
      round(all_gold['keyword_top10'].mean(), 2), 'of them; random 10:', round(all_gold['random10'].mean(), 2))

122 tasks: keyword fallback (issue title) finds a gold function in 0.3 of them; random 10: 0.06
